# Arbol de decision con MovieLens

**Carrera:** Ingenieria de Sistemas  
**Asignatura:** Mineria de Datos  
**Docente:** Enrique Alejandro Laurel Cossio  
**Fecha:** Septiembre de 2026

## Objetivo
Construir y evaluar un modelo de **arbol de decision** para predecir si una calificacion de MovieLens sera **alta** (`rating >= 4.0`) o **baja** (`rating < 4.0`).

Este cuaderno parte del proyecto `regresion_logistica_movielens.ipynb`, utiliza los mismos archivos y mantiene la misma variable objetivo. El algoritmo se cambia por un arbol de decision para comparar su desempeno con la regresion logistica.

## Requisitos y estructura esperada

Instala las dependencias si aun no estan disponibles:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn
```

Estructura del proyecto:

```text
proyecto/
|-- arbol_decision_movielens.ipynb
|-- regresion_logistica_movielens.ipynb
|-- ml-latest-small/
|   |-- ratings.csv
|   |-- movies.csv
|   |-- tags.csv
|   `-- links.csv
```

Las imagenes se muestran con `plt.show()` en las celdas. Por ello, despues de ejecutar y guardar el cuaderno, quedan almacenadas dentro del archivo `.ipynb`, en las salidas de las celdas. Ademas, se guardan copias PNG en `./output_arbol_decision`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
DATA_DIR = Path("./ml-latest-small")
OUTPUT_DIR = Path("./output_arbol_decision")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TARGET_THRESHOLD = 4.0
print(f"Directorio de datos: {DATA_DIR.resolve()}")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")

Directorio de datos: /home/jasser/Documentos/MINERIA2/Miner-a-de-Datos-DevBugs/ml-latest-small
Directorio de salida: /home/jasser/Documentos/MINERIA2/Miner-a-de-Datos-DevBugs/output_arbol_decision


## 1. Carga de datos

Se cargan `ratings.csv` y `movies.csv`. La columna `movieId` se utiliza para unir cada calificacion con los generos de la pelicula correspondiente.

In [2]:
ratings_path = DATA_DIR / "ratings.csv"
movies_path = DATA_DIR / "movies.csv"
if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError("No se encontraron ratings.csv y/o movies.csv. Verifica la carpeta ml-latest-small.")
ratings = pd.read_csv(ratings_path, encoding="utf-8")
movies = pd.read_csv(movies_path, encoding="utf-8")
required_ratings = {"userId", "movieId", "rating", "timestamp"}
required_movies = {"movieId", "title", "genres"}
if not required_ratings.issubset(ratings.columns):
    raise ValueError(f"Columnas faltantes en ratings.csv: {required_ratings - set(ratings.columns)}")
if not required_movies.issubset(movies.columns):
    raise ValueError(f"Columnas faltantes en movies.csv: {required_movies - set(movies.columns)}")
print(f"Dimensiones de ratings: {ratings.shape}")
print(f"Dimensiones de movies: {movies.shape}")
display(ratings.head())
display(movies.head())

Dimensiones de ratings: (100836, 4)
Dimensiones de movies: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 2. Preparacion de datos

La variable objetivo es `rating_alto`. Se crean las siguientes caracteristicas numericas:

- `genre_count`: cantidad de generos de la pelicula.
- `user_mean_rating`: promedio historico del usuario.
- `user_rating_count`: cantidad de calificaciones del usuario.
- `year`: ano registrado en el timestamp.

Para mantener la misma logica del proyecto de regresion logistica, las estadisticas del usuario se calculan antes de separar entrenamiento y prueba. Esta decision debe mencionarse como una limitacion metodologica porque puede introducir fuga de informacion.

In [ ]:
ratings_model = ratings.copy()
ratings_model["rating_alto"] = (ratings_model["rating"] >= TARGET_THRESHOLD).astype(int)
df = ratings_model.merge(movies[["movieId", "genres"]], on="movieId", how="left")
df["genres"] = df["genres"].fillna("")
df["genre_count"] = df["genres"].map(lambda value: 0 if value in ("", "(no genres listed)") else len(value.split("|")))
user_stats = df.groupby("userId")["rating"].agg(["mean", "count"]).reset_index()
user_stats.columns = ["userId", "user_mean_rating", "user_rating_count"]
df = df.merge(user_stats, on="userId", how="left")
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce", utc=True)
df["year"] = df["datetime"].dt.year
feature_cols = ["genre_count", "user_mean_rating", "user_rating_count", "year"]
X = df[feature_cols].copy()
y = df["rating_alto"].copy()
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)
print(f"Observaciones disponibles: {len(X):,}")
print("Distribucion de la variable objetivo:")
display(y.value_counts().rename(index={0: "Rating bajo", 1: "Rating alto"}).to_frame("cantidad"))
display(X.head())

## 3. Division de entrenamiento y prueba

Se utiliza el 80% de los datos para entrenar y el 20% para evaluar. `stratify=y` conserva la proporcion de ratings altos y bajos en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print(f"Registros de entrenamiento: {len(X_train):,}")
print(f"Registros de prueba: {len(X_test):,}")
print(f"Proporcion de ratings altos en entrenamiento: {y_train.mean():.2%}")
print(f"Proporcion de ratings altos en prueba: {y_test.mean():.2%}")

## 4. Entrenamiento del arbol

El arbol utiliza `max_depth=6` y `min_samples_leaf=50` para reducir el sobreajuste. `class_weight="balanced"` compensa posibles diferencias entre las clases. Tambien se entrena una linea base que siempre predice la clase mayoritaria.

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE)
baseline = DummyClassifier(strategy="most_frequent")
tree_model.fit(X_train, y_train)
baseline.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)
y_prob = tree_model.predict_proba(X_test)[:, 1]
print("Arbol de decision entrenado correctamente.")

## 5. Evaluacion del modelo

Se calculan accuracy, balanced accuracy, precision, recall, F1, ROC-AUC y average precision. La linea base permite saber si el arbol mejora frente a una estrategia que ignora las caracteristicas.

In [ ]:
def metric_row(model_name, actual, predicted, probability):
    return {"modelo": model_name, "accuracy": accuracy_score(actual, predicted), "balanced_accuracy": balanced_accuracy_score(actual, predicted), "precision": precision_score(actual, predicted, zero_division=0), "recall": recall_score(actual, predicted, zero_division=0), "f1": f1_score(actual, predicted, zero_division=0), "roc_auc": roc_auc_score(actual, probability), "average_precision": average_precision_score(actual, probability)}
metrics = pd.DataFrame([metric_row("Arbol de decision", y_test, y_pred, y_prob), metric_row("Linea base", y_test, baseline.predict(X_test), baseline.predict_proba(X_test)[:, 1])])
display(metrics.style.format({column: "{:.4f}" for column in metrics.columns if column != "modelo"}))
print("Reporte de clasificacion:")
print(classification_report(y_test, y_pred, target_names=["Rating bajo", "Rating alto"], zero_division=0))
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False, encoding="utf-8")

## 6. Matriz de confusion

La matriz muestra verdaderos negativos, falsos positivos, falsos negativos y verdaderos positivos. El grafico se muestra dentro de la celda y queda embebido en el cuaderno despues de guardarlo.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Bajo (0)", "Alto (1)"], yticklabels=["Bajo (0)", "Alto (1)"], ax=ax)
ax.set_xlabel("Prediccion")
ax.set_ylabel("Valor real")
ax.set_title("Matriz de confusion - Arbol de decision")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## 7. Curvas ROC y Precision-Recall

Las dos curvas se generan en una misma figura. Se usa `plt.show()` para que la imagen forme parte de la salida del cuaderno.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
precision, recall, _ = precision_recall_curve(y_test, y_prob)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color="darkorange", label=f"ROC-AUC = {roc_auc_score(y_test, y_prob):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="navy", label="Clasificador aleatorio")
axes[0].set_xlabel("Tasa de falsos positivos")
axes[0].set_ylabel("Tasa de verdaderos positivos")
axes[0].set_title("Curva ROC")
axes[0].legend()
axes[1].plot(recall, precision, color="purple", label=f"Average precision = {average_precision_score(y_test, y_prob):.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "curvas_clasificacion.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## 8. Visualizacion del arbol

Se muestran los primeros niveles para que el grafico sea legible. Las reglas completas se guardan en un archivo de texto.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(tree_model, feature_names=feature_cols, class_names=["Bajo", "Alto"], filled=True, rounded=True, max_depth=3, fontsize=9, ax=ax)
ax.set_title("Arbol de decision - primeros tres niveles")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "arbol_decision.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
text_tree = export_text(tree_model, feature_names=feature_cols)
(OUTPUT_DIR / "reglas_arbol_decision.txt").write_text(text_tree, encoding="utf-8")
print(text_tree[:4000])

## 9. Importancia de las variables

La importancia indica cuanto contribuye cada variable a las decisiones del arbol. Es una medida del modelo y no debe interpretarse como causalidad.

In [ ]:
importance = pd.DataFrame({"variable": feature_cols, "importancia": tree_model.feature_importances_}).sort_values("importancia", ascending=False).reset_index(drop=True)
display(importance)
importance.to_csv(OUTPUT_DIR / "importancia_variables.csv", index=False, encoding="utf-8")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=importance, x="importancia", y="variable", color="steelblue", ax=ax)
ax.set_title("Importancia de variables - Arbol de decision")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "importancia_variables.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## 10. Validacion cruzada

Se aplica validacion cruzada estratificada de cinco particiones sobre el conjunto de entrenamiento. Esto permite observar la estabilidad del modelo antes de analizar los resultados finales del conjunto de prueba.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(tree_model, X_train, y_train, cv=cv, scoring={"roc_auc": "roc_auc", "f1": "f1", "balanced_accuracy": "balanced_accuracy"}, n_jobs=-1)
cv_summary = pd.DataFrame({"metrica": ["ROC-AUC", "F1", "Balanced accuracy"], "media": [cv_results["test_roc_auc"].mean(), cv_results["test_f1"].mean(), cv_results["test_balanced_accuracy"].mean()], "desviacion": [cv_results["test_roc_auc"].std(), cv_results["test_f1"].std(), cv_results["test_balanced_accuracy"].std()]})
display(cv_summary.style.format({"media": "{:.4f}", "desviacion": "{:.4f}"}))
cv_summary.to_csv(OUTPUT_DIR / "resumen_validacion_cruzada.csv", index=False, encoding="utf-8")

## 11. Reporte automatico y conclusiones

El reporte se genera automaticamente con los resultados obtenidos al ejecutar las celdas. Para el informe academico, se recomienda interpretar las metricas junto con la matriz de confusion y la comparacion contra la regresion logistica.

In [ ]:
report_lines = ["INFORME: ARBOL DE DECISION - MOVIELENS", "=" * 60, "", "Objetivo:", "  Predecir si una calificacion es alta (>= 4.0) o baja (< 4.0).", "", f"Observaciones modeladas: {len(X):,}", f"Observaciones de entrenamiento: {len(X_train):,}", f"Observaciones de prueba: {len(X_test):,}", "", "Caracteristicas:", "  - genre_count", "  - user_mean_rating", "  - user_rating_count", "  - year", "", "Configuracion:", "  - max_depth: 6", "  - min_samples_leaf: 50", "  - class_weight: balanced", "", "Metricas del arbol en prueba:"]
for key, value in metric_row("Arbol de decision", y_test, y_pred, y_prob).items():
    if key != "modelo": report_lines.append(f"  {key}: {value:.4f}")
report_lines.extend(["", "Matriz de confusion:", str(cm), "", "Limitaciones:", "  - Los usuarios son anonimos y no hay informacion demografica.", "  - Las estadisticas del usuario se calcularon antes de la division train/test, igual que en el proyecto original; esto puede producir fuga de informacion.", "  - El dataset ml-latest-small es de desarrollo y no es un benchmark definitivo.", "  - La importancia de variables describe al modelo y no demuestra causalidad.", "", "Referencia:", "Harper, F. M., y Konstan, J. A. (2015). The MovieLens Datasets: History and Context.", "ACM Transactions on Interactive Intelligent Systems, 5(4), 19:1-19:19.", "https://doi.org/10.1145/2827872"])
report_path = OUTPUT_DIR / "reporte_arbol_decision.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Analisis completado correctamente.")
print(f"Reporte: {report_path.resolve()}")